# Image to text and audio

Generate a caption and audio from your own image with MUNI-PoE.

Use the [README environment](README.md#installation) with a CUDA GPU. Prepare `gemmae_3.pt`, `gemma-3-1b-pt/`, `audioldm-m-full/`, and `lightweight_audio_ae.pth` under `CKPT_ROOT/flowbind` as described in [Checkpoints](README.md#checkpoints). No dataset cache is required.

In [ ]:
import os
from pathlib import Path

import soundfile as sf
import torch
from diffusers.utils import logging as diffusers_logging
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from PIL import Image
from transformers.utils import logging as transformers_logging

from muni.config_utils import load_config
from muni.evaluation.checkpoints import build_muni_model_from_checkpoint
from muni.evaluation.image_text_audio.decoding import decode_generated_output
from scripts.data_prep.image_text_audio.features import FeatureEncoders

In [ ]:
transformers_logging.disable_progress_bar()
diffusers_logging.disable_progress_bar()

checkpoint_root = Path(os.environ.get('CKPT_ROOT', 'ckpts')).expanduser().resolve()
output_dir = Path('output/demo')
output_dir.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Run this demo with a CUDA GPU.'
device = torch.device('cuda')
seed = 42
print(f'Using {device}')

## Load the model

MUNI-PoE (about 1.3 GB) and CLIP weights are downloaded as needed.

In [ ]:
model_checkpoint = checkpoint_root / 'muni' / 'muni_poe.pth'
if not model_checkpoint.is_file():
    model_checkpoint = Path(hf_hub_download(
        repo_id='32V/MUNI', filename='muni_poe.pth',
        local_dir=str(checkpoint_root / 'muni'),
    ))


In [ ]:
cfg = load_config(
    'configs/image_text_audio/muni_poe.yaml',
    [f'paths.ckpt_root={checkpoint_root}', 'train.compile_backbones=false'],
    resolve=False,
)
model = build_muni_model_from_checkpoint(cfg, model_checkpoint, device)
image_encoder = FeatureEncoders(
    assets={},
    modalities={'image'}, device=str(device),
)

## Choose an image

Set `image_path` to your local image, or leave it as `None` to download the generated beach example from the [MUNI gallery](https://muni-proj.github.io/).

In [ ]:
image_path = None  # Or a local path, e.g. Path('my_photo.jpg')

if image_path is None:
    image_path = output_dir / 'example.png'
    if not image_path.is_file():
        torch.hub.download_url_to_file(
            'https://muni-proj.github.io/muni/qualitative/qualitative_ita/uncond_ours/ours_poe_200_255.png',
            str(image_path),
        )
else:
    image_path = Path(image_path).expanduser()
with Image.open(image_path) as image:
    preview = image.convert('RGB')
    preview.thumbnail((512, 512))
display(preview)

## Generate

Both outputs share one sample from the image posterior. Change `seed` for another sample, or choose a new image above and rerun from there. The caption and 16 kHz audio are saved to `output/demo/caption.txt` and `output/demo/audio.wav` (overwritten on each run).

In [ ]:
image_features = torch.from_numpy(image_encoder.encode_images([image_path])).to(device)

In [ ]:
torch.manual_seed(seed)
with torch.inference_mode():
    z = model.encoders['image'].posterior(image_features).sample()
    caption = model.decoders['text'].sample(z_cond=z, deterministic=True)[0]
    audio_features = model.decoders['audio'].sample(z_cond=z, deterministic=True)
    waveform = decode_generated_output(
        audio_features, 'audio', device=device, batch_size=1,
        checkpoint_paths=cfg.image_text_audio.checkpoints,
        decoder_name=cfg.image_text_audio.audio_decoder, show_progress=False,
    )[0]

In [ ]:
(output_dir / 'caption.txt').write_text(caption + '\n', encoding='utf-8')
sf.write(output_dir / 'audio.wav', waveform, samplerate=16000)
print(caption)
display(Audio(waveform, rate=16000))
print(f'Saved caption.txt and audio.wav in {output_dir}/')